# Supply-Prescript — Prescriptive Optimization

## 1. Prescriptive Analytics Objective

The predictive model identifies the supply-chain risk level as:

- Low Risk
- Moderate Risk
- High Risk

The purpose of this module is to convert these risk predictions into actionable
supply-chain decisions.

The prescriptive optimization model will determine the best operational action
while considering cost, delay, inventory availability, and supply-chain risk.

### Main Objective

Minimize the total operational impact of supply-chain disruptions by selecting
the most suitable mitigation action.

### Decision Process

Predictive Risk Model  
↓  
Risk Classification  
↓  
Generate Possible Mitigation Actions  
↓  
Evaluate Cost, Delay and Risk  
↓  
Optimization Model  
↓  
Select Optimal Action  
↓  
Recommended Supply-Chain Decision

## 2. Optimization Problem Definition

For shipments predicted to have supply-chain risk, the system must recommend
an appropriate mitigation strategy.

The optimization considers three possible actions:

1. **Normal Operation** — Continue with the existing supply-chain plan.
2. **Expedited Shipping** — Use faster transportation to reduce expected delay.
3. **Alternative Supplier** — Source the required material from another supplier.

Each action has a different:

- Operational cost
- Expected delay
- Risk exposure
- Mitigation effectiveness

The optimization model will select the action that produces the minimum
overall business impact while satisfying operational constraints.

In [25]:
try:
    import pulp
    print("PuLP version:", pulp.__version__)
    print("PuLP is installed successfully.")
except ImportError:
    print("PuLP is not installed.")

PuLP version: 3.3.2
PuLP is installed successfully.


In [26]:
import sys
print(sys.executable)

c:\ProgramData\anaconda3\python.exe


In [27]:
import sys
!{sys.executable} -m pip install pulp

Defaulting to user installation because normal site-packages is not writeable


In [28]:
try:
    import pulp
    print("PuLP version:", pulp.__version__)
    print("PuLP is installed successfully.")
except ImportError:
    print("PuLP is not installed.")

PuLP version: 3.3.2
PuLP is installed successfully.


## 3. Define Prescriptive Decision Alternatives

The prescriptive optimization model evaluates multiple mitigation actions for high-risk supply chain situations.

Each action has:
- an implementation cost
- a risk reduction benefit
- an operational impact

The optimizer will select the action that minimizes total business impact while reducing supply chain risk.

In [29]:
import pandas as pd
import numpy as np

# Define mitigation actions
actions = pd.DataFrame({
    "action": [
        "No Action",
        "Expedite Shipping",
        "Switch Supplier",
        "Increase Safety Stock"
    ],
    
    "action_cost": [
        0,
        300,
        500,
        250
    ],
    
    "risk_reduction": [
        0.00,
        0.25,
        0.40,
        0.20
    ],
    
    "operational_impact": [
        0,
        100,
        200,
        80
    ]
})

print("=" * 65)
print("PRESCRIPTIVE MITIGATION ACTIONS")
print("=" * 65)

display(actions)

PRESCRIPTIVE MITIGATION ACTIONS


,action,action_cost,risk_reduction,operational_impact
0,No Action,0,0.00,0
1,Expedite Shipping,300,0.25,100
2,Switch Supplier,500,0.40,200
3,Increase Safety Stock,250,0.20,80


## 4. Load the Trained Predictive Model

The trained Random Forest predictive model from the predictive modeling phase is loaded here.

This model will be integrated with the prescriptive optimization system so that predicted supply-chain risk can be converted into an optimal mitigation decision.

In [30]:
import joblib
from pathlib import Path

# Path of the trained predictive model
model_path = Path("../models/supply_chain_risk_model.pkl")

# Check whether model exists
if not model_path.exists():
    raise FileNotFoundError(f"Model not found at: {model_path}")

# Load model
risk_model = joblib.load(model_path)

print("=" * 65)
print("PREDICTIVE MODEL LOADED FOR PRESCRIPTIVE OPTIMIZATION")
print("=" * 65)

print("Model file :", model_path.name)
print("Model type :", type(risk_model).__name__)
print("Status     : Loaded successfully")

PREDICTIVE MODEL LOADED FOR PRESCRIPTIVE OPTIMIZATION
Model file : supply_chain_risk_model.pkl
Model type : Pipeline
Status     : Loaded successfully


## 5. Load Data for Prescriptive Decision Making

The cleaned supply-chain dataset is loaded to generate risk predictions for the prescriptive optimization stage.

The same input features used during predictive modeling must be maintained to ensure compatibility with the saved machine learning pipeline.

In [31]:
from pathlib import Path

processed_dir = Path("../data/processed")

print("=" * 65)
print("AVAILABLE PROCESSED DATA FILES")
print("=" * 65)

files = list(processed_dir.glob("*"))

if len(files) == 0:
    print("No files found in data/processed/")
else:
    for file in files:
        print(file.name)

AVAILABLE PROCESSED DATA FILES
.gitkeep
supply_chain_cleaned.csv


## 6. Load Cleaned Dataset and Prepare Model Inputs

The cleaned supply-chain dataset is loaded and the same pre-decision features used during predictive modeling are selected for risk prediction.

In [32]:
import pandas as pd

data_path = "../data/processed/supply_chain_cleaned.csv"

df = pd.read_csv(data_path)

print("=" * 65)
print("CLEANED DATASET LOADED SUCCESSFULLY")
print("=" * 65)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

CLEANED DATASET LOADED SUCCESSFULLY
Shape: (113097, 18)

Columns:
['warehouse_inventory_level', 'handling_equipment_availability', 'order_fulfillment_status', 'weather_condition_severity', 'shipping_costs', 'supplier_reliability_score', 'lead_time_days', 'historical_demand', 'cargo_condition_status', 'route_risk_level', 'customs_clearance_time', 'disruption_likelihood_score', 'delay_probability', 'risk_classification', 'delivery_time_deviation', 'product_id', 'supplier_id', 'supplier_country']


In [33]:
# Exact features used by the final production model

model_features = [
    "warehouse_inventory_level",
    "handling_equipment_availability",
    "weather_condition_severity",
    "shipping_costs",
    "supplier_reliability_score",
    "lead_time_days",
    "historical_demand",
    "route_risk_level",
    "customs_clearance_time",
    "supplier_country"
]

X_prescriptive = df[model_features].copy()

print("=" * 65)
print("MODEL INPUT DATA PREPARED")
print("=" * 65)

print("Number of rows:", X_prescriptive.shape[0])
print("Number of features:", X_prescriptive.shape[1])

print("\nFeatures:")
for i, col in enumerate(X_prescriptive.columns, 1):
    print(f"{i}. {col}")

X_prescriptive.head()

MODEL INPUT DATA PREPARED
Number of rows: 113097
Number of features: 10

Features:
1. warehouse_inventory_level
2. handling_equipment_availability
3. weather_condition_severity
4. shipping_costs
5. supplier_reliability_score
6. lead_time_days
7. historical_demand
8. route_risk_level
9. customs_clearance_time
10. supplier_country


,warehouse_inventory_level,handling_equipment_availability,weather_condition_severity,shipping_costs,supplier_reliability_score,lead_time_days,historical_demand,route_risk_level,customs_clearance_time,supplier_country
0,985.716862,0.481294,0.359066,456.503853,0.986064,2.128009,100.772854,1.182116,0.502006,Greece
1,985.716862,0.481294,0.359066,456.503853,0.986064,2.128009,100.772854,1.182116,0.502006,South Korea
2,985.716862,0.481294,0.359066,456.503853,0.986064,2.128009,100.772854,1.182116,0.502006,Ethiopia
3,396.700206,0.620780,0.230660,640.408205,0.463233,12.608166,5313.738114,9.611988,0.966774,Iran
4,396.700206,0.620780,0.230660,640.408205,0.463233,12.608166,5313.738114,9.611988,0.966774,Guatemala


## 7. Generate Supply Chain Risk Predictions

The trained predictive model is applied to the prepared supply-chain records to generate risk classifications.

For each record, the model produces:
- Predicted risk class
- Prediction probability
- Prediction confidence

These predictions will serve as the risk input for the prescriptive optimization model.

In [34]:
# Generate risk predictions
predicted_risk = risk_model.predict(X_prescriptive)

# Generate class probabilities
risk_probabilities = risk_model.predict_proba(X_prescriptive)

# Highest probability = prediction confidence
prediction_confidence = risk_probabilities.max(axis=1)

print("=" * 65)
print("RISK PREDICTIONS GENERATED")
print("=" * 65)

print("Records predicted :", len(predicted_risk))
print("Model classes      :", risk_model.classes_)

print("\nPredicted Risk Distribution:")
print(pd.Series(predicted_risk).value_counts())

print("\nAverage Prediction Confidence:")
print(round(prediction_confidence.mean(), 4))

RISK PREDICTIONS GENERATED
Records predicted : 113097
Model classes      : ['High Risk' 'Low Risk' 'Moderate Risk']

Predicted Risk Distribution:
High Risk        84368
Moderate Risk    17783
Low Risk         10946
Name: count, dtype: int64

Average Prediction Confidence:
0.9381


In [35]:
# Create prescriptive decision dataset
prescriptive_df = df.copy()

prescriptive_df["predicted_risk"] = predicted_risk
prescriptive_df["prediction_confidence"] = prediction_confidence

print("=" * 65)
print("PRESCRIPTIVE DECISION DATASET CREATED")
print("=" * 65)

print("Dataset shape:", prescriptive_df.shape)

display(
    prescriptive_df[
        [
            "predicted_risk",
            "prediction_confidence"
        ]
    ].head(10)
)

PRESCRIPTIVE DECISION DATASET CREATED
Dataset shape: (113097, 20)


,predicted_risk,prediction_confidence
0,Moderate Risk,0.860000
1,Moderate Risk,0.883333
2,Moderate Risk,0.890000
3,High Risk,0.953333
4,High Risk,0.966667
5,High Risk,0.940000
6,High Risk,0.960000
7,High Risk,0.956667
8,High Risk,0.986667
9,High Risk,0.973333


## 8. Convert Predicted Risk into Numerical Risk Score

The predicted supply-chain risk categories are converted into numerical values so that they can be used by the prescriptive optimization model.

Risk mapping:

- Low Risk = 1
- Moderate Risk = 2
- High Risk = 3

In [37]:
# Create prescriptive decision dataset
# ============================================================
# STEP 6: PREPARE MODEL INPUT FOR PRESCRIPTIVE DECISION MAKING
# ============================================================

model_features = [
    "warehouse_inventory_level",
    "handling_equipment_availability",
    "weather_condition_severity",
    "shipping_costs",
    "supplier_reliability_score",
    "lead_time_days",
    "historical_demand",
    "route_risk_level",
    "customs_clearance_time",
    "supplier_country"
]

# Check that all required columns exist
missing_features = [
    col for col in model_features
    if col not in df.columns
]

if missing_features:
    print("Missing columns:", missing_features)
else:
    model_input = df[model_features].copy()

    print("=" * 65)
    print("MODEL INPUT DATA PREPARED")
    print("=" * 65)

    print("Number of rows:", model_input.shape[0])
    print("Number of features:", model_input.shape[1])

    print("\nFeatures:")
    for i, column in enumerate(model_input.columns, start=1):
        print(f"{i}. {column}")

    print("\nModel input shape:", model_input.shape)

MODEL INPUT DATA PREPARED
Number of rows: 113097
Number of features: 10

Features:
1. warehouse_inventory_level
2. handling_equipment_availability
3. weather_condition_severity
4. shipping_costs
5. supplier_reliability_score
6. lead_time_days
7. historical_demand
8. route_risk_level
9. customs_clearance_time
10. supplier_country

Model input shape: (113097, 10)


In [38]:
# Create prescriptive decision dataset

decision_df = df.copy()

decision_df["predicted_risk"] = risk_model.predict(model_input)

prediction_probabilities = risk_model.predict_proba(model_input)

decision_df["prediction_confidence"] = prediction_probabilities.max(axis=1)

print("=" * 65)
print("PRESCRIPTIVE DECISION DATASET CREATED")
print("=" * 65)

print("Dataset shape:", decision_df.shape)

display(
    decision_df[
        ["predicted_risk", "prediction_confidence"]
    ].head(10)
)

PRESCRIPTIVE DECISION DATASET CREATED
Dataset shape: (113097, 20)


,predicted_risk,prediction_confidence
0,Moderate Risk,0.860000
1,Moderate Risk,0.883333
2,Moderate Risk,0.890000
3,High Risk,0.953333
4,High Risk,0.966667
5,High Risk,0.940000
6,High Risk,0.960000
7,High Risk,0.956667
8,High Risk,0.986667
9,High Risk,0.973333


In [39]:
# Convert predicted risk categories into numeric risk scores

risk_score_mapping = {
    "Low Risk": 1,
    "Moderate Risk": 2,
    "High Risk": 3
}

decision_df["risk_score"] = decision_df["predicted_risk"].map(
    risk_score_mapping
)

print("=" * 65)
print("NUMERICAL RISK SCORES CREATED")
print("=" * 65)

display(
    decision_df[
        [
            "predicted_risk",
            "prediction_confidence",
            "risk_score"
        ]
    ].head(10)
)

NUMERICAL RISK SCORES CREATED


,predicted_risk,prediction_confidence,risk_score
0,Moderate Risk,0.860000,2
1,Moderate Risk,0.883333,2
2,Moderate Risk,0.890000,2
3,High Risk,0.953333,3
4,High Risk,0.966667,3
5,High Risk,0.940000,3
6,High Risk,0.960000,3
7,High Risk,0.956667,3
8,High Risk,0.986667,3
9,High Risk,0.973333,3


## 9. Calculate Risk Exposure Score

The predicted risk level is combined with model confidence to create a numerical risk exposure score.

This score represents the severity and confidence of the predicted supply-chain risk and will be used by the prescriptive optimization model to evaluate mitigation actions.

In [40]:
# ============================================================
# STEP 9: CALCULATE RISK EXPOSURE SCORE
# ============================================================

decision_df["risk_exposure"] = (
    decision_df["risk_score"] *
    decision_df["prediction_confidence"]
)

print("=" * 65)
print("RISK EXPOSURE SCORE CREATED")
print("=" * 65)

print("\nRisk Exposure Summary:")
print(
    decision_df["risk_exposure"]
    .describe()
    .round(4)
)

print("\nSample Records:")

display(
    decision_df[
        [
            "predicted_risk",
            "risk_score",
            "prediction_confidence",
            "risk_exposure"
        ]
    ].head(10)
)

RISK EXPOSURE SCORE CREATED

Risk Exposure Summary:
count    113097.0000
mean          2.5101
std           0.6805
min           0.6533
25%           1.8867
50%           2.8700
75%           2.9100
max           3.0000
Name: risk_exposure, dtype: float64

Sample Records:


,predicted_risk,risk_score,prediction_confidence,risk_exposure
0,Moderate Risk,2,0.860000,1.720000
1,Moderate Risk,2,0.883333,1.766667
2,Moderate Risk,2,0.890000,1.780000
3,High Risk,3,0.953333,2.860000
4,High Risk,3,0.966667,2.900000
5,High Risk,3,0.940000,2.820000
6,High Risk,3,0.960000,2.880000
7,High Risk,3,0.956667,2.870000
8,High Risk,3,0.986667,2.960000
9,High Risk,3,0.973333,2.920000


## 10. Evaluate Mitigation Actions

Each available mitigation action is evaluated against the predicted supply-chain risk.

For every action, the model calculates the remaining risk exposure after applying the action's risk-reduction effectiveness.

Remaining Risk = Risk Exposure × (1 - Risk Reduction)

This allows the prescriptive system to compare the effectiveness of different mitigation strategies before optimization.

In [41]:
# ============================================================
# STEP 10: EVALUATE MITIGATION ACTIONS
# ============================================================

action_evaluations = []

for _, action_row in actions.iterrows():

    action_name = action_row["action"]
    action_cost = action_row["action_cost"]
    risk_reduction = action_row["risk_reduction"]
    operational_impact = action_row["operational_impact"]

    # Calculate remaining risk for all records
    remaining_risk = (
        decision_df["risk_exposure"] *
        (1 - risk_reduction)
    )

    action_evaluations.append({
        "action": action_name,
        "action_cost": action_cost,
        "risk_reduction": risk_reduction,
        "operational_impact": operational_impact,
        "avg_original_risk": decision_df["risk_exposure"].mean(),
        "avg_remaining_risk": remaining_risk.mean(),
        "avg_risk_reduced": (
            decision_df["risk_exposure"].mean()
            - remaining_risk.mean()
        )
    })

action_evaluation_df = pd.DataFrame(action_evaluations)

print("=" * 70)
print("MITIGATION ACTION EVALUATION")
print("=" * 70)

display(
    action_evaluation_df.round(4)
)

MITIGATION ACTION EVALUATION


,action,action_cost,risk_reduction,operational_impact,avg_original_risk,avg_remaining_risk,avg_risk_reduced
0,No Action,0,0.00,0,2.5101,2.5101,0.0000
1,Expedite Shipping,300,0.25,100,2.5101,1.8826,0.6275
2,Switch Supplier,500,0.40,200,2.5101,1.5061,1.0040
3,Increase Safety Stock,250,0.20,80,2.5101,2.0081,0.5020


## 11. Define Optimization Objective

The prescriptive optimization model selects the best mitigation action by balancing:

- Remaining supply-chain risk
- Cost of the mitigation action
- Operational impact

The optimization objective is to minimize the total business impact.

Objective Score =
Remaining Risk + Normalized Action Cost + Normalized Operational Impact

Weights are used to represent the relative importance of each component.

In [42]:
# ============================================================
# STEP 11: DEFINE OPTIMIZATION OBJECTIVE
# ============================================================

# Importance weights
RISK_WEIGHT = 0.60
COST_WEIGHT = 0.25
OPERATIONAL_WEIGHT = 0.15

# Maximum values used for normalization
max_cost = actions["action_cost"].max()
max_operational_impact = actions["operational_impact"].max()

# Avoid division by zero
if max_cost == 0:
    max_cost = 1

if max_operational_impact == 0:
    max_operational_impact = 1


# Normalize values
action_evaluation_df["normalized_cost"] = (
    action_evaluation_df["action_cost"] / max_cost
)

action_evaluation_df["normalized_operational_impact"] = (
    action_evaluation_df["operational_impact"]
    / max_operational_impact
)

# Normalize remaining risk to approximately 0–1
max_risk = action_evaluation_df["avg_original_risk"].max()

action_evaluation_df["normalized_remaining_risk"] = (
    action_evaluation_df["avg_remaining_risk"] / max_risk
)


# Calculate total objective score
action_evaluation_df["objective_score"] = (

    RISK_WEIGHT
    * action_evaluation_df["normalized_remaining_risk"]

    + COST_WEIGHT
    * action_evaluation_df["normalized_cost"]

    + OPERATIONAL_WEIGHT
    * action_evaluation_df["normalized_operational_impact"]
)


print("=" * 75)
print("OPTIMIZATION OBJECTIVE SCORES")
print("=" * 75)

print("\nObjective Weights:")
print("Risk Weight        :", RISK_WEIGHT)
print("Cost Weight        :", COST_WEIGHT)
print("Operational Weight :", OPERATIONAL_WEIGHT)

print("\nAction Evaluation:")

display(
    action_evaluation_df[
        [
            "action",
            "avg_remaining_risk",
            "normalized_remaining_risk",
            "normalized_cost",
            "normalized_operational_impact",
            "objective_score"
        ]
    ].round(4)
)

OPTIMIZATION OBJECTIVE SCORES

Objective Weights:
Risk Weight        : 0.6
Cost Weight        : 0.25
Operational Weight : 0.15

Action Evaluation:


,action,avg_remaining_risk,normalized_remaining_risk,normalized_cost,normalized_operational_impact,objective_score
0,No Action,2.5101,1.00,0.0,0.0,0.600
1,Expedite Shipping,1.8826,0.75,0.6,0.5,0.675
2,Switch Supplier,1.5061,0.60,1.0,1.0,0.760
3,Increase Safety Stock,2.0081,0.80,0.5,0.4,0.665


## 12. Define Business Constraints

Business constraints ensure that the optimization model recommends practical mitigation actions.

Rules:

- Low Risk → No Action is acceptable.
- Moderate Risk → A mitigation action should be considered.
- High Risk → A mitigation action is mandatory.
- The selected action should reduce supply-chain risk.
- Only one mitigation action will be selected for a decision.

In [43]:
# ============================================================
# STEP 12: DEFINE BUSINESS CONSTRAINTS
# ============================================================

# Minimum risk reduction required for each risk category
minimum_risk_reduction = {
    "Low Risk": 0.00,
    "Moderate Risk": 0.20,
    "High Risk": 0.25
}

print("=" * 70)
print("BUSINESS CONSTRAINTS DEFINED")
print("=" * 70)

print("\nMinimum Required Risk Reduction:")

for risk, reduction in minimum_risk_reduction.items():
    print(f"{risk:<15} : {reduction:.0%}")

print("\nDecision Rules:")
print("1. Low Risk      -> No Action is allowed.")
print("2. Moderate Risk -> At least 20% risk reduction required.")
print("3. High Risk     -> At least 25% risk reduction required.")
print("4. Only one mitigation action can be selected.")
print("5. Objective     -> Minimize total business impact.")

BUSINESS CONSTRAINTS DEFINED

Minimum Required Risk Reduction:
Low Risk        : 0%
Moderate Risk   : 20%
High Risk       : 25%

Decision Rules:
1. Low Risk      -> No Action is allowed.
2. Moderate Risk -> At least 20% risk reduction required.
3. High Risk     -> At least 25% risk reduction required.
4. Only one mitigation action can be selected.
5. Objective     -> Minimize total business impact.


## 13. Build PuLP Optimization Model

The optimization model selects the best mitigation action while minimizing the total business impact.

The model uses:

- Binary decision variables
- Objective scores calculated earlier
- Minimum risk-reduction constraints
- Exactly one action must be selected

In [45]:
# ============================================================
# STEP 13: BUILD PULP OPTIMIZATION MODEL
# ============================================================

import pulp

# Current scenario
current_risk_class = "High Risk"

# Get required risk reduction
required_reduction = minimum_risk_reduction[current_risk_class]

print("=" * 70)
print("BUILDING OPTIMIZATION MODEL")
print("=" * 70)

print("Risk class              :", current_risk_class)
print("Required risk reduction :", required_reduction)


# ------------------------------------------------------------
# Create minimization problem
# ------------------------------------------------------------

optimization_model = pulp.LpProblem(
    "Supply_Chain_Mitigation_Optimization",
    pulp.LpMinimize
)


# ------------------------------------------------------------
# Create binary decision variables
# ------------------------------------------------------------

decision_variables = {}

for i, row in action_evaluation_df.iterrows():

    action_name = row["action"]

    safe_name = (
        action_name
        .replace(" ", "_")
        .replace("-", "_")
    )

    decision_variables[action_name] = pulp.LpVariable(
        f"select_{safe_name}",
        cat="Binary"
    )


print("\nDecision variables created:")

for action_name in decision_variables:
    print("-", action_name)


# ------------------------------------------------------------
# Objective Function
# ------------------------------------------------------------

optimization_model += pulp.lpSum(

    action_evaluation_df.loc[
        action_evaluation_df["action"] == action_name,
        "objective_score"
    ].iloc[0]

    * decision_variables[action_name]

    for action_name in decision_variables
)


# ------------------------------------------------------------
# Constraint 1:
# Exactly one action must be selected
# ------------------------------------------------------------

optimization_model += (

    pulp.lpSum(
        decision_variables.values()
    ) == 1,

    "Select_Exactly_One_Action"
)


# ------------------------------------------------------------
# Constraint 2:
# Minimum required risk reduction
# ------------------------------------------------------------

optimization_model += (

    pulp.lpSum(

        action_evaluation_df.loc[
            action_evaluation_df["action"] == action_name,
            "risk_reduction"
        ].iloc[0]

        * decision_variables[action_name]

        for action_name in decision_variables

    ) >= required_reduction,

    "Minimum_Risk_Reduction"
)


print("\n" + "=" * 70)
print("OPTIMIZATION MODEL CREATED SUCCESSFULLY")
print("=" * 70)

print("\nModel Name:")
print(optimization_model.name)

print("\nNumber of Variables:")
print(len(optimization_model.variables()))

print("\nNumber of Constraints:")
print(len(optimization_model.constraints))

BUILDING OPTIMIZATION MODEL
Risk class              : High Risk
Required risk reduction : 0.25

Decision variables created:
- No Action
- Expedite Shipping
- Switch Supplier
- Increase Safety Stock

OPTIMIZATION MODEL CREATED SUCCESSFULLY

Model Name:
Supply_Chain_Mitigation_Optimization

Number of Variables:
4

Number of Constraints:
2


## 14. Solve the Prescriptive Optimization Model

The optimization model is solved using the PuLP CBC solver.

The selected action must satisfy the required minimum risk reduction while minimizing the overall business objective score.

In [46]:
# ============================================================
# STEP 14: SOLVE THE OPTIMIZATION MODEL
# ============================================================

# Solve using PuLP's default CBC solver
solver = pulp.PULP_CBC_CMD(msg=False)

optimization_model.solve(solver)

print("=" * 70)
print("OPTIMIZATION RESULT")
print("=" * 70)

print("Solver Status :", pulp.LpStatus[optimization_model.status])
print("Objective Value :", round(pulp.value(optimization_model.objective), 4))

print("\nSelected Action:")

selected_action = None

for action_name, variable in decision_variables.items():
    if variable.value() == 1:
        selected_action = action_name
        print(action_name)

print("\nDecision Variable Values:")

for action_name, variable in decision_variables.items():
    print(f"{action_name}: {variable.value()}")

OPTIMIZATION RESULT
Solver Status : Optimal
Objective Value : 0.675

Selected Action:
Expedite Shipping

Decision Variable Values:
No Action: 0.0
Expedite Shipping: 1.0
Switch Supplier: 0.0
Increase Safety Stock: 0.0


In [47]:
# Get full details of the recommended action

recommended_action_details = action_evaluation_df[
    action_evaluation_df["action"] == selected_action
].copy()

print("=" * 70)
print("RECOMMENDED MITIGATION ACTION DETAILS")
print("=" * 70)

display(
    recommended_action_details[
        [
            "action",
            "action_cost",
            "risk_reduction",
            "operational_impact",
            "avg_original_risk",
            "avg_remaining_risk",
            "avg_risk_reduced",
            "objective_score"
        ]
    ]
)

RECOMMENDED MITIGATION ACTION DETAILS


,action,action_cost,risk_reduction,operational_impact,avg_original_risk,avg_remaining_risk,avg_risk_reduced,objective_score
1,Expedite Shipping,300,0.25,100,2.510124,1.882593,0.627531,0.675


## 15. Generate Record-Level Prescriptive Recommendations

The predictive model identifies the supply chain risk level, while the
prescriptive layer recommends an appropriate mitigation action.

Decision rules:

- Low Risk → No Action
- Moderate Risk → Select an action providing at least 20% risk reduction
- High Risk → Select an action providing at least 25% risk reduction

Among feasible actions, the action with the lowest objective score is selected.

In [48]:
# ============================================================
# STEP 15: GENERATE RECORD-LEVEL PRESCRIPTIVE RECOMMENDATIONS
# ============================================================

# Create a copy to preserve the original decision dataset
prescriptive_df = decision_df.copy()

# Risk reduction requirements
risk_requirements = {
    "Low Risk": 0.00,
    "Moderate Risk": 0.20,
    "High Risk": 0.25
}

# Function to select the best mitigation action
def recommend_action(risk_class):

    required_reduction = risk_requirements.get(risk_class, 0)

    # Low-risk cases require no mitigation
    if risk_class == "Low Risk":
        return "No Action"

    # Find actions satisfying required risk reduction
    feasible_actions = action_evaluation_df[
        action_evaluation_df["risk_reduction"] >= required_reduction
    ]

    # Select action with minimum objective score
    best_action = feasible_actions.loc[
        feasible_actions["objective_score"].idxmin()
    ]

    return best_action["action"]


# Generate recommendations
prescriptive_df["recommended_action"] = (
    prescriptive_df["predicted_risk"].apply(recommend_action)
)

print("=" * 70)
print("PRESCRIPTIVE RECOMMENDATIONS GENERATED")
print("=" * 70)

print("\nTotal records:", len(prescriptive_df))

print("\nRecommendation Distribution:")
print(
    prescriptive_df["recommended_action"]
    .value_counts()
)

print("\nSample Recommendations:")

display(
    prescriptive_df[
        [
            "predicted_risk",
            "prediction_confidence",
            "risk_score",
            "risk_exposure",
            "recommended_action"
        ]
    ].head(15)
)

PRESCRIPTIVE RECOMMENDATIONS GENERATED

Total records: 113097

Recommendation Distribution:
recommended_action
Expedite Shipping        84368
Increase Safety Stock    17783
No Action                10946
Name: count, dtype: int64

Sample Recommendations:


,predicted_risk,prediction_confidence,risk_score,risk_exposure,recommended_action
0,Moderate Risk,0.860000,2,1.720000,Increase Safety Stock
1,Moderate Risk,0.883333,2,1.766667,Increase Safety Stock
2,Moderate Risk,0.890000,2,1.780000,Increase Safety Stock
3,High Risk,0.953333,3,2.860000,Expedite Shipping
4,High Risk,0.966667,3,2.900000,Expedite Shipping
5,High Risk,0.940000,3,2.820000,Expedite Shipping
6,High Risk,0.960000,3,2.880000,Expedite Shipping
7,High Risk,0.956667,3,2.870000,Expedite Shipping
8,High Risk,0.986667,3,2.960000,Expedite Shipping
9,High Risk,0.973333,3,2.920000,Expedite Shipping


## 16. Calculate Expected Cost and Remaining Risk

This step attaches the business impact of each recommended action to every
supply-chain record.

For each recommendation, we calculate:

- Action cost
- Risk reduction percentage
- Operational impact
- Expected risk reduction
- Expected remaining risk

This converts the recommended action into measurable business outcomes.

In [49]:
# ============================================================
# STEP 16: CALCULATE BUSINESS IMPACT OF EACH RECOMMENDATION
# ============================================================

# Create lookup dictionaries from action definitions
cost_map = actions.set_index("action")["action_cost"].to_dict()
reduction_map = actions.set_index("action")["risk_reduction"].to_dict()
impact_map = actions.set_index("action")["operational_impact"].to_dict()

# Attach action information to every recommendation
prescriptive_df["recommended_action_cost"] = (
    prescriptive_df["recommended_action"].map(cost_map)
)

prescriptive_df["recommended_risk_reduction"] = (
    prescriptive_df["recommended_action"].map(reduction_map)
)

prescriptive_df["recommended_operational_impact"] = (
    prescriptive_df["recommended_action"].map(impact_map)
)

# Calculate expected amount of risk removed
prescriptive_df["expected_risk_reduced"] = (
    prescriptive_df["risk_exposure"]
    * prescriptive_df["recommended_risk_reduction"]
)

# Calculate remaining risk after applying recommendation
prescriptive_df["expected_remaining_risk"] = (
    prescriptive_df["risk_exposure"]
    - prescriptive_df["expected_risk_reduced"]
)

print("=" * 75)
print("BUSINESS IMPACT OF PRESCRIPTIVE RECOMMENDATIONS")
print("=" * 75)

print("\nTotal Records:", len(prescriptive_df))

print(
    "\nTotal Expected Action Cost:",
    prescriptive_df["recommended_action_cost"].sum()
)

print(
    "Average Original Risk Exposure:",
    round(prescriptive_df["risk_exposure"].mean(), 4)
)

print(
    "Average Expected Remaining Risk:",
    round(prescriptive_df["expected_remaining_risk"].mean(), 4)
)

print(
    "Average Expected Risk Reduced:",
    round(prescriptive_df["expected_risk_reduced"].mean(), 4)
)

print("\nSample Prescriptive Results:")

display(
    prescriptive_df[
        [
            "predicted_risk",
            "prediction_confidence",
            "risk_exposure",
            "recommended_action",
            "recommended_action_cost",
            "recommended_risk_reduction",
            "expected_risk_reduced",
            "expected_remaining_risk"
        ]
    ].head(15)
)

BUSINESS IMPACT OF PRESCRIPTIVE RECOMMENDATIONS

Total Records: 113097

Total Expected Action Cost: 29756150
Average Original Risk Exposure: 2.5101
Average Expected Remaining Risk: 1.9172
Average Expected Risk Reduced: 0.5929

Sample Prescriptive Results:


,predicted_risk,prediction_confidence,risk_exposure,recommended_action,recommended_action_cost,recommended_risk_reduction,expected_risk_reduced,expected_remaining_risk
0,Moderate Risk,0.860000,1.720000,Increase Safety Stock,250,0.20,0.344000,1.376000
1,Moderate Risk,0.883333,1.766667,Increase Safety Stock,250,0.20,0.353333,1.413333
2,Moderate Risk,0.890000,1.780000,Increase Safety Stock,250,0.20,0.356000,1.424000
3,High Risk,0.953333,2.860000,Expedite Shipping,300,0.25,0.715000,2.145000
4,High Risk,0.966667,2.900000,Expedite Shipping,300,0.25,0.725000,2.175000
5,High Risk,0.940000,2.820000,Expedite Shipping,300,0.25,0.705000,2.115000
6,High Risk,0.960000,2.880000,Expedite Shipping,300,0.25,0.720000,2.160000
7,High Risk,0.956667,2.870000,Expedite Shipping,300,0.25,0.717500,2.152500
8,High Risk,0.986667,2.960000,Expedite Shipping,300,0.25,0.740000,2.220000
9,High Risk,0.973333,2.920000,Expedite Shipping,300,0.25,0.730000,2.190000


## 17. Final Prescriptive KPI Summary

This step summarizes the overall performance of the prescriptive decision
system, including risk distribution, recommended actions, expected cost,
and expected risk reduction.

In [50]:
# ============================================================
# STEP 17: FINAL PRESCRIPTIVE KPI SUMMARY
# ============================================================

total_records = len(prescriptive_df)

high_risk_count = (prescriptive_df["predicted_risk"] == "High Risk").sum()
moderate_risk_count = (prescriptive_df["predicted_risk"] == "Moderate Risk").sum()
low_risk_count = (prescriptive_df["predicted_risk"] == "Low Risk").sum()

total_cost = prescriptive_df["recommended_action_cost"].sum()

avg_original_risk = prescriptive_df["risk_exposure"].mean()

avg_remaining_risk = prescriptive_df["expected_remaining_risk"].mean()

avg_risk_reduced = prescriptive_df["expected_risk_reduced"].mean()

# Overall percentage reduction
overall_risk_reduction_pct = (
    (avg_original_risk - avg_remaining_risk)
    / avg_original_risk
) * 100


print("=" * 75)
print("FINAL PRESCRIPTIVE ANALYTICS KPI SUMMARY")
print("=" * 75)

print(f"\nTotal Supply Chain Records : {total_records:,}")

print("\nRISK DISTRIBUTION")
print("-" * 40)

print(f"Low Risk      : {low_risk_count:,}")
print(f"Moderate Risk : {moderate_risk_count:,}")
print(f"High Risk     : {high_risk_count:,}")


print("\nRECOMMENDATION DISTRIBUTION")
print("-" * 40)

print(
    prescriptive_df["recommended_action"]
    .value_counts()
    .to_string()
)


print("\nBUSINESS IMPACT")
print("-" * 40)

print(f"Total Expected Action Cost     : {total_cost:,.2f}")

print(
    f"Average Original Risk Exposure : "
    f"{avg_original_risk:.4f}"
)

print(
    f"Average Remaining Risk         : "
    f"{avg_remaining_risk:.4f}"
)

print(
    f"Average Risk Reduced           : "
    f"{avg_risk_reduced:.4f}"
)

print(
    f"Overall Expected Risk Reduction: "
    f"{overall_risk_reduction_pct:.2f}%"
)


print("\nACTION-WISE BUSINESS SUMMARY")
print("-" * 75)

action_summary = (
    prescriptive_df
    .groupby("recommended_action")
    .agg(
        total_records=("recommended_action", "size"),
        total_action_cost=("recommended_action_cost", "sum"),
        avg_original_risk=("risk_exposure", "mean"),
        avg_remaining_risk=("expected_remaining_risk", "mean"),
        avg_risk_reduced=("expected_risk_reduced", "mean")
    )
    .reset_index()
)

display(action_summary)

FINAL PRESCRIPTIVE ANALYTICS KPI SUMMARY

Total Supply Chain Records : 113,097

RISK DISTRIBUTION
----------------------------------------
Low Risk      : 10,946
Moderate Risk : 17,783
High Risk     : 84,368

RECOMMENDATION DISTRIBUTION
----------------------------------------
recommended_action
Expedite Shipping        84368
Increase Safety Stock    17783
No Action                10946

BUSINESS IMPACT
----------------------------------------
Total Expected Action Cost     : 29,756,150.00
Average Original Risk Exposure : 2.5101
Average Remaining Risk         : 1.9172
Average Risk Reduced           : 0.5929
Overall Expected Risk Reduction: 23.62%

ACTION-WISE BUSINESS SUMMARY
---------------------------------------------------------------------------


,recommended_action,total_records,total_action_cost,avg_original_risk,avg_remaining_risk,avg_risk_reduced
0,Expedite Shipping,84368,25310400,2.884822,2.163616,0.721205
1,Increase Safety Stock,17783,4445750,1.747171,1.397737,0.349434
2,No Action,10946,0,0.861587,0.861587,0.000000


## 18. Export Final Prescriptive Results

This step saves the final prescriptive analytics results and action-wise
business summary for database integration, reporting, and closed-loop
decision tracking.

In [51]:
# ============================================================
# STEP 18: EXPORT FINAL PRESCRIPTIVE RESULTS
# ============================================================

from pathlib import Path

# Project root
project_root = Path.cwd()

# If notebook is running from notebooks folder
if project_root.name == "notebooks":
    project_root = project_root.parent

# Processed data folder
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Save complete prescriptive recommendations
# ------------------------------------------------------------

prescriptive_file = output_dir / "prescriptive_recommendations.csv"

prescriptive_df.to_csv(
    prescriptive_file,
    index=False
)

# ------------------------------------------------------------
# 2. Save action-wise KPI summary
# ------------------------------------------------------------

summary_file = output_dir / "prescriptive_action_summary.csv"

action_summary.to_csv(
    summary_file,
    index=False
)

# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

print("=" * 75)
print("PRESCRIPTIVE ANALYTICS RESULTS EXPORTED")
print("=" * 75)

print("\n1. Final Prescriptive Dataset")
print("File :", prescriptive_file.name)
print("Rows :", len(prescriptive_df))
print("Columns :", len(prescriptive_df.columns))

print("\n2. Action Summary")
print("File :", summary_file.name)
print("Rows :", len(action_summary))

print("\nSaved Location:")
print(output_dir)

print("\nExport Status: SUCCESS")

PRESCRIPTIVE ANALYTICS RESULTS EXPORTED

1. Final Prescriptive Dataset
File : prescriptive_recommendations.csv
Rows : 113097
Columns : 28

2. Action Summary
File : prescriptive_action_summary.csv
Rows : 3

Saved Location:
c:\Users\Mansi Kurne\Desktop\Supply-Prescript\data\processed

Export Status: SUCCESS


## 19. Advanced Optimization with Business Constraints

The existing prescriptive analytics engine is enhanced by adding practical business constraints such as budget, time, and operational capacity. These constraints ensure that the recommended actions are feasible for real-world supply chain operations.

In [1]:
advanced_actions = {
    "No Action": {
        "cost": 0,
        "risk_reduction": 0.00,
        "time_saved_days": 0,
        "capacity_required": 0,
        "operational_impact": 0
    },

    "Expedite Shipping": {
        "cost": 300,
        "risk_reduction": 0.25,
        "time_saved_days": 7,
        "capacity_required": 20,
        "operational_impact": 100
    },

    "Switch Supplier": {
        "cost": 500,
        "risk_reduction": 0.40,
        "time_saved_days": 10,
        "capacity_required": 35,
        "operational_impact": 200
    },

    "Increase Safety Stock": {
        "cost": 250,
        "risk_reduction": 0.20,
        "time_saved_days": 4,
        "capacity_required": 15,
        "operational_impact": 80
    }
}

for action, details in advanced_actions.items():
    print(action, ":", details)

No Action : {'cost': 0, 'risk_reduction': 0.0, 'time_saved_days': 0, 'capacity_required': 0, 'operational_impact': 0}
Expedite Shipping : {'cost': 300, 'risk_reduction': 0.25, 'time_saved_days': 7, 'capacity_required': 20, 'operational_impact': 100}
Switch Supplier : {'cost': 500, 'risk_reduction': 0.4, 'time_saved_days': 10, 'capacity_required': 35, 'operational_impact': 200}
Increase Safety Stock : {'cost': 250, 'risk_reduction': 0.2, 'time_saved_days': 4, 'capacity_required': 15, 'operational_impact': 80}


### 19.1 Define Business Constraints

In [2]:
MAX_BUDGET = 600
MIN_TIME_SAVING = 4
MAX_CAPACITY = 40

print("Maximum Budget:", MAX_BUDGET)
print("Minimum Time Saving:", MIN_TIME_SAVING, "days")
print("Maximum Capacity:", MAX_CAPACITY)

Maximum Budget: 600
Minimum Time Saving: 4 days
Maximum Capacity: 40


### 19.2 Check Feasible Actions

In [3]:
feasible_actions = []

for action, details in advanced_actions.items():

    budget_ok = details["cost"] <= MAX_BUDGET
    capacity_ok = details["capacity_required"] <= MAX_CAPACITY

    time_ok = (
        action == "No Action"
        or details["time_saved_days"] >= MIN_TIME_SAVING
    )

    if budget_ok and capacity_ok and time_ok:
        feasible_actions.append(action)

print("Feasible Actions")
print("----------------")

for action in feasible_actions:
    print(action)

Feasible Actions
----------------
No Action
Expedite Shipping
Switch Supplier
Increase Safety Stock


### 19.3 Rank Feasible Mitigation Actions

The feasible mitigation actions are evaluated using a weighted business score based on risk reduction, action cost, time saving, capacity usage, and operational impact. A lower optimization score represents a more cost-effective and operationally feasible mitigation strategy.

In [4]:
# Define weights for the multi-objective optimization

RISK_WEIGHT = 0.40
COST_WEIGHT = 0.25
TIME_WEIGHT = 0.20
CAPACITY_WEIGHT = 0.10
OPERATIONAL_WEIGHT = 0.05

ranked_actions = []

for action in feasible_actions:

    details = advanced_actions[action]

    # Normalize values
    normalized_cost = details["cost"] / MAX_BUDGET
    normalized_capacity = details["capacity_required"] / MAX_CAPACITY

    # Higher time saving is beneficial
    normalized_time_benefit = details["time_saved_days"] / 10

    # Higher risk reduction is beneficial
    risk_benefit = details["risk_reduction"]

    # Normalize operational impact
    normalized_operational_impact = details["operational_impact"] / 200

    # Lower score = better action
    optimization_score = (
        COST_WEIGHT * normalized_cost
        + CAPACITY_WEIGHT * normalized_capacity
        + OPERATIONAL_WEIGHT * normalized_operational_impact
        - RISK_WEIGHT * risk_benefit
        - TIME_WEIGHT * normalized_time_benefit
    )

    ranked_actions.append({
        "action": action,
        "cost": details["cost"],
        "risk_reduction": details["risk_reduction"],
        "time_saved_days": details["time_saved_days"],
        "capacity_required": details["capacity_required"],
        "operational_impact": details["operational_impact"],
        "optimization_score": round(optimization_score, 4)
    })

ranked_actions

[{'action': 'No Action',
  'cost': 0,
  'risk_reduction': 0.0,
  'time_saved_days': 0,
  'capacity_required': 0,
  'operational_impact': 0,
  'optimization_score': 0.0},
 {'action': 'Expedite Shipping',
  'cost': 300,
  'risk_reduction': 0.25,
  'time_saved_days': 7,
  'capacity_required': 20,
  'operational_impact': 100,
  'optimization_score': -0.04},
 {'action': 'Switch Supplier',
  'cost': 500,
  'risk_reduction': 0.4,
  'time_saved_days': 10,
  'capacity_required': 35,
  'operational_impact': 200,
  'optimization_score': -0.0142},
 {'action': 'Increase Safety Stock',
  'cost': 250,
  'risk_reduction': 0.2,
  'time_saved_days': 4,
  'capacity_required': 15,
  'operational_impact': 80,
  'optimization_score': 0.0017}]

### 19.4 Generate Top 3 Prescriptive Alternatives

For disruption scenarios requiring intervention, the "No Action" option is excluded from the mitigation ranking. The remaining feasible mitigation strategies are ranked according to their optimization scores.

In [5]:
# Keep only actual mitigation actions

mitigation_rankings = [
    item for item in ranked_actions
    if item["action"] != "No Action"
]

# Sort from lowest optimization score to highest
mitigation_rankings = sorted(
    mitigation_rankings,
    key=lambda x: x["optimization_score"]
)

# Select Top 3
top_3_actions = mitigation_rankings[:3]

print("=" * 75)
print("TOP 3 PRESCRIPTIVE ALTERNATIVES")
print("=" * 75)

for rank, item in enumerate(top_3_actions, start=1):

    print(f"\nRank {rank}: {item['action']}")
    print(f"Cost               : {item['cost']}")
    print(f"Risk Reduction     : {item['risk_reduction'] * 100:.0f}%")
    print(f"Time Saved         : {item['time_saved_days']} days")
    print(f"Capacity Required  : {item['capacity_required']}")
    print(f"Operational Impact : {item['operational_impact']}")
    print(f"Optimization Score : {item['optimization_score']}")

TOP 3 PRESCRIPTIVE ALTERNATIVES

Rank 1: Expedite Shipping
Cost               : 300
Risk Reduction     : 25%
Time Saved         : 7 days
Capacity Required  : 20
Operational Impact : 100
Optimization Score : -0.04

Rank 2: Switch Supplier
Cost               : 500
Risk Reduction     : 40%
Time Saved         : 10 days
Capacity Required  : 35
Operational Impact : 200
Optimization Score : -0.0142

Rank 3: Increase Safety Stock
Cost               : 250
Risk Reduction     : 20%
Time Saved         : 4 days
Capacity Required  : 15
Operational Impact : 80
Optimization Score : 0.0017


### 19.5 Create Top 3 Recommendation Dataset

The ranked prescriptive alternatives are converted into a structured dataset so that they can later be stored in the database and used by the operational application.

In [6]:
import pandas as pd

top_3_df = pd.DataFrame(top_3_actions)

# Add ranking column
top_3_df.insert(
    0,
    "recommendation_rank",
    range(1, len(top_3_df) + 1)
)

top_3_df

,recommendation_rank,action,cost,risk_reduction,time_saved_days,capacity_required,operational_impact,optimization_score
0,1,Expedite Shipping,300,0.25,7,20,100,-0.0400
1,2,Switch Supplier,500,0.40,10,35,200,-0.0142
2,3,Increase Safety Stock,250,0.20,4,15,80,0.0017


### 19.6 Validate Business Constraints

Each recommended alternative is validated against the maximum budget, minimum time-saving requirement, and maximum operational capacity. This provides an optimization audit to ensure that infeasible actions are never recommended.

In [7]:
top_3_df["budget_valid"] = (
    top_3_df["cost"] <= MAX_BUDGET
)

top_3_df["time_valid"] = (
    top_3_df["time_saved_days"] >= MIN_TIME_SAVING
)

top_3_df["capacity_valid"] = (
    top_3_df["capacity_required"] <= MAX_CAPACITY
)

top_3_df["all_constraints_satisfied"] = (
    top_3_df["budget_valid"]
    & top_3_df["time_valid"]
    & top_3_df["capacity_valid"]
)

top_3_df[
    [
        "recommendation_rank",
        "action",
        "budget_valid",
        "time_valid",
        "capacity_valid",
        "all_constraints_satisfied"
    ]
]

,recommendation_rank,action,budget_valid,time_valid,capacity_valid,all_constraints_satisfied
0,1,Expedite Shipping,True,True,True,True
1,2,Switch Supplier,True,True,True,True
2,3,Increase Safety Stock,True,True,True,True


In [8]:
if top_3_df["all_constraints_satisfied"].all():

    print("=" * 65)
    print("OPTIMIZATION AUDIT: PASSED")
    print("=" * 65)
    print("All Top 3 recommendations satisfy:")
    print(f"✓ Budget <= {MAX_BUDGET}")
    print(f"✓ Time Saving >= {MIN_TIME_SAVING} days")
    print(f"✓ Capacity Required <= {MAX_CAPACITY}")

else:
    print("OPTIMIZATION AUDIT: FAILED")
    print("One or more recommendations violate business constraints.")

OPTIMIZATION AUDIT: PASSED
All Top 3 recommendations satisfy:
✓ Budget <= 600
✓ Time Saving >= 4 days
✓ Capacity Required <= 40


In [9]:
from pathlib import Path

output_path = Path("../data/processed/top_3_prescriptive_alternatives.csv")

top_3_df.to_csv(
    output_path,
    index=False
)

print("Top 3 Prescriptive Alternatives Exported Successfully")
print("File:", output_path)
print("Rows:", len(top_3_df))
print("Columns:", len(top_3_df.columns))

Top 3 Prescriptive Alternatives Exported Successfully
File: ..\data\processed\top_3_prescriptive_alternatives.csv
Rows: 3
Columns: 12
